In [3]:
import numpy as np
import torch
from datasets import load_dataset, concatenate_datasets
from transformers import AutoTokenizer, DataCollatorWithPadding, AutoModelForSequenceClassification, set_seed, TrainingArguments, Trainer
from sklearn.metrics import accuracy_score, f1_score, classification_report

In [4]:
if torch.cuda.is_available():
    print("CUDA is available. Using GPU.")
else:
    print("CUDA is not available. Using CPU.")

CUDA is available. Using GPU.


In [5]:
MODEL_NAME = "cointegrated/rubert-tiny2"

In [6]:
NUM_LABELS = 3  # 0=negative, 1=neutral, 2=positive

In [7]:
ds = load_dataset('./../datasets/reviews_cleaned')

In [8]:
ds

DatasetDict({
    train: Dataset({
        features: ['Unnamed: 0.1', 'Unnamed: 0', 'language', 'rating', 'category', 'combined_text'],
        num_rows: 119048
    })
})

In [9]:
set(ds['train']['language'])

{'kazakh', 'other', 'russian'}

In [10]:
ds_russian = ds.filter(lambda x: x['language'] in ('russian'))

In [11]:
def is_valid(example):
    t = example["combined_text"]
    if t is None:
        return False
    if isinstance(t, float) and np.isnan(t):
        return False
    if isinstance(t, str) and t.strip() == "":
        return False
    return True

ds_russian = ds_russian.filter(is_valid)

In [12]:
ds_russian

DatasetDict({
    train: Dataset({
        features: ['Unnamed: 0.1', 'Unnamed: 0', 'language', 'rating', 'category', 'combined_text'],
        num_rows: 111667
    })
})

In [13]:
def calc_sentinent(row):
    sentiment = 1
    if row['rating'] > 3.0:
        sentiment = 2
    elif row['rating'] < 3.0:
        sentiment = 0
    else:
        sentiment = 1
    return {'label': sentiment}

In [14]:
ds_russian = ds_russian.map(calc_sentinent)

In [15]:
ds_russian['train']

Dataset({
    features: ['Unnamed: 0.1', 'Unnamed: 0', 'language', 'rating', 'category', 'combined_text', 'label'],
    num_rows: 111667
})

In [16]:
num_negative = ds_russian['train'].filter(lambda row: row['label'] == 0).num_rows
num_neutral = ds_russian['train'].filter(lambda row: row['label'] == 1).num_rows
num_positive = ds_russian['train'].filter(lambda row: row['label'] == 2).num_rows

In [17]:
print('Number of negative reviews: {}'.format(num_negative))
print('Number of neutral reviews: {}'.format(num_neutral))
print('Number of positive reviews: {}'.format(num_positive))

Number of negative reviews: 3788
Number of neutral reviews: 3227
Number of positive reviews: 104652


In [18]:
df_russian_negative = ds_russian['train'].filter(lambda row: row['label'] == 0).shuffle(seed=42).select(range(num_negative))
df_russian_neutral = ds_russian['train'].filter(lambda row: row['label'] == 1).shuffle(seed=42).select(range(num_neutral))
df_russian_positive = ds_russian['train'].filter(lambda row: row['label'] == 2).shuffle(seed=42).select(range(num_negative)) #YES, we reduce numbers

In [19]:
df_russian_reduced = concatenate_datasets([df_russian_negative, df_russian_neutral, df_russian_positive]).shuffle(seed=42)

In [20]:
df_russian_reduced

Dataset({
    features: ['Unnamed: 0.1', 'Unnamed: 0', 'language', 'rating', 'category', 'combined_text', 'label'],
    num_rows: 10803
})

In [21]:
ds_russian_splitted = df_russian_reduced.train_test_split(test_size=0.1, seed=42)

In [22]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

In [23]:
def tokenize(batch):
    return tokenizer(batch['combined_text'], truncation=True, max_length=256)

In [24]:
ds_tok = ds_russian_splitted.map(tokenize, batched=True)

In [25]:
ds_tok = ds_tok.remove_columns([c for c in ds_tok["train"].column_names if c not in ("input_ids", "attention_mask", "label")])

In [26]:
data_collator = DataCollatorWithPadding(tokenizer=tokenizer,
                                        pad_to_multiple_of=8 if torch.cuda.is_available() else None
                                        )

In [27]:
id2label = {0: "negative", 1: "neutral", 2: "positive"}
label2id = {v: k for k, v in id2label.items()}

In [28]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_LABELS,
    id2label=id2label,
    label2id=label2id,
    # ignore_mismatched_sizes=True,  # включите, если ругается на размер классификационной головы
)

Loading weights:   0%|          | 0/55 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cointegrated/rubert-tiny2
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
bert.embeddings.position_ids               | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider trai

In [29]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1_macro": f1_score(labels, preds, average="macro"),
    }


In [30]:
set_seed(42)

In [31]:
args = TrainingArguments(
    output_dir="./../sentiment_bge_m3_ru_cls",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=3,
    weight_decay=0.01,
    warmup_ratio=0.05,
    save_steps=50,
    eval_steps=50,
    eval_strategy="steps",
    save_strategy="steps",
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    greater_is_better=True,
    logging_steps=50,
    fp16=torch.cuda.is_available(),
    report_to="none",
    lr_scheduler_type="linear",  # или "cosine"
    gradient_accumulation_steps=2,  # 16*2=32 эффективный train batch
    max_grad_norm=1.0,
)

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


In [32]:
trainer = Trainer(
    model=model,
    args=args,
    train_dataset=ds_tok["train"],
    eval_dataset=ds_tok["test"],
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)


In [33]:
trainer.train()

Step,Training Loss,Validation Loss,Accuracy,F1 Macro
50,2.157391,1.048707,0.524514,0.494418
100,1.995924,0.920274,0.587419,0.515984
150,1.754555,0.804635,0.671600,0.629585
200,1.602244,0.742612,0.673451,0.625636
250,1.507766,0.710509,0.679001,0.641116
300,1.468242,0.688986,0.687327,0.651406
350,1.443377,0.679340,0.688252,0.667209
400,1.377459,0.670030,0.693802,0.660036
450,1.415805,0.660460,0.702128,0.681442
500,1.383975,0.654438,0.703053,0.682810


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.beta', 'bert.embeddings.LayerNorm.gamma', 'bert.encoder.layer.0.attention.output.LayerNorm.beta', 'bert.encoder.layer.0.attention.output.LayerNorm.gamma', 'bert.

TrainOutput(global_step=912, training_loss=1.4868042970958508, metrics={'train_runtime': 35.9721, 'train_samples_per_second': 810.796, 'train_steps_per_second': 25.353, 'total_flos': 62294130362592.0, 'train_loss': 1.4868042970958508, 'epoch': 3.0})

In [34]:
trainer.save_model("./../sentiment_balanced_ruberta_cls/prod")
tokenizer.save_pretrained("./../sentiment_balanced_ruberta_cls/prod")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('./../sentiment_balanced_ruberta_cls/prod/tokenizer_config.json',
 './../sentiment_balanced_ruberta_cls/prod/tokenizer.json')

In [35]:
pred = trainer.predict(ds_tok["test"])
val_preds = np.argmax(pred.predictions, axis=-1)

print("Metrics:", pred.metrics)
print(classification_report(ds_tok["test"]["label"], val_preds, target_names=[id2label[i] for i in range(NUM_LABELS)]))

Metrics: {'test_loss': 0.6462286114692688, 'test_accuracy': 0.7067530064754857, 'test_f1_macro': 0.6911896972738227, 'test_runtime': 0.3691, 'test_samples_per_second': 2928.944, 'test_steps_per_second': 92.122}
              precision    recall  f1-score   support

    negative       0.69      0.80      0.74       395
     neutral       0.56      0.46      0.50       319
    positive       0.84      0.83      0.83       367

    accuracy                           0.71      1081
   macro avg       0.69      0.69      0.69      1081
weighted avg       0.70      0.71      0.70      1081



In [36]:
@torch.inference_mode()
def predict_sentiment(texts, max_length=256):
    model.eval()
    device = next(model.parameters()).device
    batch = tokenizer(texts, return_tensors="pt", padding=True, truncation=True, max_length=max_length)
    batch = {k: v.to(device) for k, v in batch.items()}
    out = model(**batch)
    probs = torch.softmax(out.logits, dim=-1).cpu().numpy()
    pred_ids = probs.argmax(axis=-1)
    return [
        {
            "label": model.config.id2label[int(i)],
            "scores": {model.config.id2label[j]: float(p[j]) for j in range(probs.shape[1])},
        }
        for i, p in zip(pred_ids, probs)
    ]


In [37]:
print(predict_sentiment(["Товар отличный, доставка быстрая", "Полный ужас, не советую"]))

[{'label': 'positive', 'scores': {'negative': 0.016074836254119873, 'neutral': 0.07308764010667801, 'positive': 0.9108374714851379}}, {'label': 'negative', 'scores': {'negative': 0.7645480036735535, 'neutral': 0.17914114892482758, 'positive': 0.056310802698135376}}]
